In [1]:
import gymnasium as gym
import pickle
import numpy as np

## Parameters

In [ ]:
alpha = 0.1
gamma = 0.99
epsilon = 1.0 
epsilon_min = 0.01
epsilon_decay = 0.9995
episodes = 5000

We split in 21 to apply discretization

In [ ]:
bins_x = np.linspace(-1.0, 1.0, 20)
bins_y = np.linspace(-1.0, 1.0, 20)
bins_vel = np.linspace(-8.0, 8.0, 20)

actions_continues = np.linspace(-2.0, 2.0, 11)
nb_actions = len(actions_continues)

q_table = np.zeros((len(bins_x) + 1, len(bins_y) + 1, len(bins_vel) + 1, nb_actions))


$Q \in \mathbb{R}^{4}$ with $x = 21$, $y = 21$, $\text{vel} = 21$, $N = 11$

## Discretization

In [ ]:
def discretize_state(obs):
    x_idx = np.digitize(obs[0], bins_x)
    y_idx = np.digitize(obs[1], bins_y)
    vel_idx = np.digitize(obs[2], bins_vel)
    return (x_idx, y_idx, vel_idx)

## Training

$$Q(s_t, a_t) = (1 - \alpha) Q(s_t, a_t) + \alpha [R_t + \gamma \max_{a} Q(s_{t+1}, a_t)]$$

In [ ]:
env = gym.make("Pendulum-v1", render_mode=None)

for episode in range(episodes):
    obs, info = env.reset()
    state = discretize_state(obs)
    done = False
    truncated = False
    
    while not (done or truncated):
        m = np.random.uniform(0,1)
        if m < epsilon:
            action_idx = np.random.randint(0, nb_actions)
        else:
            action_idx = np.argmax(q_table[state])
            
        # L'environnement s'attend à recevoir une action continue sous forme de ndarray(1,)
        action_continue = np.array([actions_continues[action_idx]], dtype=np.float32)
        
        next_obs, reward, done, truncated, info = env.step(action_continue)
        next_state = discretize_state(next_obs)
        
        # Mise à jour de la Q-table (Équation de Bellman pour Q-learning)
        best_next_action = np.argmax(q_table[next_state])
        td_target = reward + gamma * q_table[next_state][best_next_action]
        q_table[state][action_idx] = (1 - alpha) * q_table[state][action_idx] + alpha * td_target
        
        state = next_state
        
    # Réduction progressive de l'exploration
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
        
    if (episode + 1) % 500 == 0:
        print(f"Épisode {episode + 1}/{episodes} terminé.")
        
env.close()

# Sauvegarde de la Q-table comme demandé dans le laboratoire
with open("pendulum.pkl", "wb") as f:
    pickle.dump(q_table, f)
print("Training finish")


## Testing

In [ ]:
# Chargement de la Q-table optimale
with open("pendulum.pkl", "rb") as f:
    q_table_loaded = pickle.load(f)
    
# Animation activée (render_mode="human")
env = gym.make("Pendulum-v1", render_mode="human")

print("Début de la visualisation...")
# On teste sur quelques épisodes
for episode in range(1):
    obs, info = env.reset()
    state = discretize_state(obs)
    done = False
    truncated = False
    total_reward = 0
    
    while not (done or truncated):
        # En test, on exploite à 100% (pas de hasard)
        action_idx = np.argmax(q_table_loaded[state])
        action_continue = np.array([actions_continues[action_idx]], dtype=np.float32)
        
        next_obs, reward, done, truncated, info = env.step(action_continue)
        state = discretize_state(next_obs)
        total_reward += reward
        
    print(f"Test Épisode {episode + 1} - Récompense Totale : {total_reward:.2f}")
    
env.close()
